# 00 Framework Tutorial

Bu notebook yeni kullanıcı için **DataHub → ETL → feature → strategy → backtest → risk → execution → portfolio → report** akışını anlamlı örneklerle gösterir. Ayrıca source recommendation örnekleri ile hangi veri kaynağını seçmeniz gerektiğini de gösterir.


In [ ]:
from algotradeplan.data import DataHub, ETL
from src.algotradeplan.orchestration.trade_flow import TradeFlow
from src.algotradeplan.plugins.connectors.simulated_fill_connector import SimulatedFillExecutionConnectorPlugin
from src.algotradeplan.plugins.risk.engine import RiskEngine
from src.algotradeplan.plugins.strategies.ema_cross_atr_stop import EmaCrossAtrStopStrategyPlugin
from src.algotradeplan.portfolio.manager import PortfolioManager

hub = DataHub()
coverage = hub.coverage_table()
summary = hub.source_summary("offline_fallback")
recommendations = {
    "crypto_spot_kline": hub.recommend_sources("crypto_spot_kline", allow_api_key=False),
    "crypto_perp_funding": hub.recommend_sources("crypto_perp_funding", allow_api_key=False),
    "macro_indicators": hub.recommend_sources("macro_indicators", allow_api_key=False),
    "public_news": hub.recommend_sources("public_news", allow_api_key=False),
}
best_equity_no_key = hub.best_sources_for(dataset="kline", asset_class="equity", allow_api_key=False)
coingecko_explain = hub.explain_source("coingecko")
funding_explain = hub.explain_dataset("funding")
datasets = hub.available_datasets("offline_fallback", implemented_only=True)
ingest = hub.ingest(source="offline_fallback", symbol="BTCUSDT", datasets=datasets, allow_partial=True)

etl = ETL(hub)
frame = etl.load_market_data(source="offline_fallback", symbol="BTCUSDT", dataset="kline", limit=120)
rows = frame.to_dict(orient="records") if hasattr(frame, "to_dict") else frame
strategy = EmaCrossAtrStopStrategyPlugin()
flow = TradeFlow(strategy=strategy, risk=RiskEngine(max_notional=2_000.0), execution=SimulatedFillExecutionConnectorPlugin())
portfolio = PortfolioManager(starting_cash=10_000.0)
result = flow.run({"symbol": "BTCUSDT", "price": float(rows[-1]["close"]), "quantity": 0.01, "candles": rows[-60:]})
portfolio_snapshot = portfolio.apply_execution(result.execution)
{"coverage_preview": coverage[:2], "summary": summary, "recommendations": recommendations, "best_equity_no_key": best_equity_no_key, "coingecko_explain": {"source": coingecko_explain["source"], "notes": coingecko_explain["notes"]}, "funding_explain": {"dataset": funding_explain["dataset"], "best_sources_no_api_key": funding_explain["best_sources_no_api_key"]}, "signal": result.signal, "risk": result.risk_decision, "execution": result.execution, "portfolio": portfolio_snapshot, "ledger": portfolio.ledger()}
